# EPA Station Monitoring + Climate Data Merge

**Goal:** Merge EPA water quality measurements with ISU climate data.

**Join keys:**
- **Location:** Each EPA monitoring location (lat/lon) is spatially matched to the nearest ISU climate station using a haversine nearest-neighbor search.
- **Time:** `ActivityStartDate` (EPA WQ) == `day` (climate).

**Input files:**
- `data/tabular/water-quality/raw/epa-wq.csv`
- `data/tabular/water-quality/raw/epa-stations.csv`
- `data/tabular/climate/raw/isu-climate.csv`

**Output:** `data/tabular/merged/epa-climate-merged.csv`

In [1]:
import numpy as np
import pandas as pd
import requests
import json
import os
from scipy.spatial import cKDTree

## Step 1: Load and Clean EPA Water Quality Data

In [2]:
wq_path = '../../data/tabular/water-quality/raw/epa-wq.csv'
df_wq = pd.read_csv(wq_path, low_memory=False)
print(f'epa-wq shape: {df_wq.shape}')

# Select key columns
wq_cols = [
    'ActivityIdentifier',
    'ActivityStartDate',
    'MonitoringLocationIdentifier',
    'MonitoringLocationName',
    'CharacteristicName',
    'ResultMeasureValue',
    'ResultMeasure/MeasureUnitCode',
    'ResultValueTypeName',
]
wq_cols = [c for c in wq_cols if c in df_wq.columns]
df_wq = df_wq[wq_cols].copy()

# Fix types
df_wq['ActivityStartDate'] = pd.to_datetime(df_wq['ActivityStartDate'], errors='coerce')
df_wq['ResultMeasureValue'] = pd.to_numeric(df_wq['ResultMeasureValue'], errors='coerce')

# Drop rows missing key fields
df_wq.dropna(subset=['MonitoringLocationIdentifier', 'ActivityStartDate', 'CharacteristicName', 'ResultMeasureValue'], inplace=True)

print(f'epa-wq cleaned shape: {df_wq.shape}')
df_wq.head(3)

epa-wq shape: (118604, 81)
epa-wq cleaned shape: (69482, 8)


,ActivityIdentifier,ActivityStartDate,MonitoringLocationIdentifier,MonitoringLocationName,CharacteristicName,ResultMeasureValue,ResultMeasure/MeasureUnitCode,ResultValueTypeName
0,nwisia.01.02400517,2024-01-01,USGS-05474000,NaN,"Temperature, water",0.0,deg C,Actual
1,nwisia.01.02400517,2024-01-01,USGS-05474000,NaN,Number of sampling points,1.0,count,Actual
2,nwisia.01.02400551,2024-02-26,USGS-05451210,NaN,Stream width measure,30.0,ft,Actual


## Step 2: Load and Clean EPA Stations Data

In [3]:
stations_path = '../../data/tabular/water-quality/raw/epa-stations.csv'
df_stations = pd.read_csv(stations_path, low_memory=False)
print(f'epa-stations shape: {df_stations.shape}')

# Select key columns
station_cols = [
    'MonitoringLocationIdentifier',
    'MonitoringLocationName',
    'MonitoringLocationTypeName',
    'HUCEightDigitCode',
    'LatitudeMeasure',
    'LongitudeMeasure',
    'StateCode',
    'CountyCode',
]
station_cols = [c for c in station_cols if c in df_stations.columns]
df_stations = df_stations[station_cols].copy()

# Fix types
df_stations['LatitudeMeasure'] = pd.to_numeric(df_stations['LatitudeMeasure'], errors='coerce')
df_stations['LongitudeMeasure'] = pd.to_numeric(df_stations['LongitudeMeasure'], errors='coerce')

# Drop stations without coordinates
df_stations.dropna(subset=['LatitudeMeasure', 'LongitudeMeasure'], inplace=True)
df_stations.drop_duplicates(subset='MonitoringLocationIdentifier', inplace=True)

print(f'epa-stations cleaned shape: {df_stations.shape}')
df_stations.head(3)

epa-stations shape: (621, 37)
epa-stations cleaned shape: (621, 8)


,MonitoringLocationIdentifier,MonitoringLocationName,MonitoringLocationTypeName,HUCEightDigitCode,LatitudeMeasure,LongitudeMeasure,StateCode,CountyCode
0,USGS-05420500,"Mississippi River at Clinton, IA",Stream,7080101,41.780586,-90.252073,19,45
1,USGS-05451210,"South Fork Iowa River NE of New Providence, IA",Stream,7080207,42.315306,-93.152194,19,83
2,USGS-05465500,"Iowa River at Wapello, IA",Stream,7080209,41.178086,-91.182092,19,115


## Step 3: Merge EPA WQ + EPA Stations on MonitoringLocationIdentifier

In [4]:
df_epa = df_wq.merge(df_stations, on='MonitoringLocationIdentifier', how='left', suffixes=('', '_station'))

# Resolve MonitoringLocationName conflict (prefer WQ value, fall back to station)
if 'MonitoringLocationName_station' in df_epa.columns:
    df_epa['MonitoringLocationName'] = df_epa['MonitoringLocationName'].fillna(df_epa['MonitoringLocationName_station'])
    df_epa.drop(columns=['MonitoringLocationName_station'], inplace=True)

print(f'EPA merged shape: {df_epa.shape}')
print(f'Rows with lat/lon: {df_epa["LatitudeMeasure"].notna().sum():,}')
df_epa.head(3)

EPA merged shape: (69482, 14)
Rows with lat/lon: 69,482


,ActivityIdentifier,ActivityStartDate,MonitoringLocationIdentifier,MonitoringLocationName,CharacteristicName,ResultMeasureValue,ResultMeasure/MeasureUnitCode,ResultValueTypeName,MonitoringLocationTypeName,HUCEightDigitCode,LatitudeMeasure,LongitudeMeasure,StateCode,CountyCode
0,nwisia.01.02400517,2024-01-01,USGS-05474000,"Skunk River at Augusta, IA","Temperature, water",0.0,deg C,Actual,Stream,7080107,40.753650,-91.277089,19,57
1,nwisia.01.02400517,2024-01-01,USGS-05474000,"Skunk River at Augusta, IA",Number of sampling points,1.0,count,Actual,Stream,7080107,40.753650,-91.277089,19,57
2,nwisia.01.02400551,2024-02-26,USGS-05451210,"South Fork Iowa River NE of New Providence, IA",Stream width measure,30.0,ft,Actual,Stream,7080207,42.315306,-93.152194,19,83


## Step 4: Fetch ISU Climate Station Coordinates from IEM

ISU climate stations are identified by codes (e.g. `IA1533`). We fetch their lat/lon from the IEM GeoJSON endpoint and filter to only stations present in our climate dataset.

In [5]:
climate_path = '../../data/tabular/climate/raw/isu-climate.csv'
df_climate = pd.read_csv(climate_path)
print(f'Climate shape: {df_climate.shape}')
print(f'Unique climate stations: {df_climate["station"].nunique()}')

# Fix date type
df_climate['day'] = pd.to_datetime(df_climate['day'], errors='coerce')
df_climate.dropna(subset=['day'], inplace=True)

climate_stations_in_data = set(df_climate['station'].unique())
print(f'Date range: {df_climate["day"].min().date()} to {df_climate["day"].max().date()}')

Climate shape: (115602, 12)
Unique climate stations: 144
Date range: 2024-01-01 to 2026-03-13


In [6]:
# Fetch station metadata from IEM
IEM_URL = 'https://mesonet.agron.iastate.edu/geojson/network/IACLIMATE.geojson'

response = requests.get(IEM_URL, timeout=30)
response.raise_for_status()
geojson = response.json()

climate_meta_rows = []
for feature in geojson['features']:
    sid = feature['id']
    lon, lat = feature['geometry']['coordinates']
    name = feature['properties'].get('sname', '')
    climate_meta_rows.append({'climate_station': sid, 'climate_station_name': name, 'climate_lat': lat, 'climate_lon': lon})

df_climate_meta = pd.DataFrame(climate_meta_rows)

# Keep only stations present in the climate dataset
df_climate_meta = df_climate_meta[df_climate_meta['climate_station'].isin(climate_stations_in_data)].reset_index(drop=True)

print(f'Climate stations with coordinates: {len(df_climate_meta)}')
df_climate_meta.head(5)

Climate stations with coordinates: 144


,climate_station,climate_station_name,climate_lat,climate_lon
0,IA0000,Iowa Average,41.7500,-93.2500
1,IA0133,ALGONA,43.0700,-94.3000
2,IA0149,Allerton,40.7036,-93.3633
3,IA0157,ALLISON,42.7500,-92.7800
4,IA0197,AMES MUNICIPAL AP,41.9904,-93.6185


## Step 5: Spatial Nearest-Neighbor — Map Each EPA Station to Closest Climate Station

We use `scipy.spatial.cKDTree` with haversine-projected coordinates to find the nearest climate station for each EPA monitoring location.

In [7]:
def haversine_deg_to_rad(lat_lon_deg):
    """Convert (lat, lon) degrees to radians scaled for haversine."""
    return np.radians(lat_lon_deg)

# Build KD-tree from climate station coordinates (in radians for haversine approx)
climate_coords_rad = np.radians(df_climate_meta[['climate_lat', 'climate_lon']].values)
tree = cKDTree(climate_coords_rad)

# Get unique EPA locations with lat/lon
df_epa_locs = df_epa[['MonitoringLocationIdentifier', 'LatitudeMeasure', 'LongitudeMeasure']].dropna().drop_duplicates()
print(f'Unique EPA locations with coordinates: {len(df_epa_locs)}')

epa_coords_rad = np.radians(df_epa_locs[['LatitudeMeasure', 'LongitudeMeasure']].values)

# Query nearest climate station for each EPA location
distances, indices = tree.query(epa_coords_rad, k=1)

# Earth radius ≈ 6371 km; convert radian distance to km
distances_km = distances * 6371

df_epa_locs = df_epa_locs.copy()
df_epa_locs['climate_station'] = df_climate_meta['climate_station'].iloc[indices].values
df_epa_locs['climate_station_name'] = df_climate_meta['climate_station_name'].iloc[indices].values
df_epa_locs['distance_to_climate_station_km'] = distances_km

print(f'\nNearest-neighbor distance stats (km):')
print(df_epa_locs['distance_to_climate_station_km'].describe().round(2))
df_epa_locs.head(5)

Unique EPA locations with coordinates: 621

Nearest-neighbor distance stats (km):
count    621.00
mean      15.14
std        8.53
min        0.17
25%        8.66
50%       14.70
75%       20.36
max       48.16
Name: distance_to_climate_station_km, dtype: float64


,MonitoringLocationIdentifier,LatitudeMeasure,LongitudeMeasure,climate_station,climate_station_name,distance_to_climate_station_km
0,USGS-05474000,40.753650,-91.277089,IA1060,BURLINGTON,14.179409
2,USGS-05451210,42.315306,-93.152194,IA4142,IOWA-FALLS,27.230696
89,USGS-05490500,40.727806,-91.959614,IA4389,KEOSAUQUA,1.444669
126,USGS-05465500,41.178086,-91.182092,IA1731,COLUMBUS JUNCT 2 SSW,22.372336
160,USGS-05420500,41.780586,-90.252073,IA1635,CLINTON-1,2.938298


## Step 6: Join Climate Station Mapping Back to EPA Data

In [8]:
# Merge climate station assignment into main EPA dataframe
df_epa = df_epa.merge(
    df_epa_locs[['MonitoringLocationIdentifier', 'climate_station', 'climate_station_name', 'distance_to_climate_station_km']],
    on='MonitoringLocationIdentifier',
    how='left'
)

print(f'EPA data with climate station assigned: {df_epa["climate_station"].notna().sum():,} / {len(df_epa):,} rows')
df_epa[['MonitoringLocationIdentifier', 'ActivityStartDate', 'climate_station', 'climate_station_name']].head(5)

EPA data with climate station assigned: 69,482 / 69,482 rows


,MonitoringLocationIdentifier,ActivityStartDate,climate_station,climate_station_name
0,USGS-05474000,2024-01-01,IA1060,BURLINGTON
1,USGS-05474000,2024-01-01,IA1060,BURLINGTON
2,USGS-05451210,2024-02-26,IA4142,IOWA-FALLS
3,USGS-05451210,2024-02-26,IA4142,IOWA-FALLS
4,USGS-05451210,2024-02-26,IA4142,IOWA-FALLS


## Step 7: Merge EPA Data with Climate Data on (climate_station, date)

In [9]:
# Normalize date columns to date-only (no time component) for joining
df_epa['merge_date'] = df_epa['ActivityStartDate'].dt.normalize()
df_climate['merge_date'] = df_climate['day'].dt.normalize()

# Rename climate station column to match
df_climate_merge = df_climate.rename(columns={'station': 'climate_station'})

# Select climate columns to bring in
climate_feature_cols = ['climate_station', 'merge_date', 'doy', 'gdd_40_86', 'high', 'highc', 'low', 'lowc', 'precip', 'snow', 'snowd']
climate_feature_cols = [c for c in climate_feature_cols if c in df_climate_merge.columns]
df_climate_features = df_climate_merge[climate_feature_cols].drop_duplicates(subset=['climate_station', 'merge_date'])

# Merge on (climate_station, merge_date)
df_merged = df_epa.merge(
    df_climate_features,
    on=['climate_station', 'merge_date'],
    how='left'
)

# Drop helper column
df_merged.drop(columns=['merge_date'], inplace=True)

print(f'Final merged shape: {df_merged.shape}')
climate_match_rate = df_merged['high'].notna().sum() / len(df_merged) * 100
print(f'Climate match rate: {climate_match_rate:.1f}%')
df_merged.head(3)

Final merged shape: (69482, 26)
Climate match rate: 99.3%


,ActivityIdentifier,ActivityStartDate,MonitoringLocationIdentifier,MonitoringLocationName,CharacteristicName,ResultMeasureValue,ResultMeasure/MeasureUnitCode,ResultValueTypeName,MonitoringLocationTypeName,HUCEightDigitCode,...,distance_to_climate_station_km,doy,gdd_40_86,high,highc,low,lowc,precip,snow,snowd
0,nwisia.01.02400517,2024-01-01,USGS-05474000,"Skunk River at Augusta, IA","Temperature, water",0.0,deg C,Actual,Stream,7080107,...,14.179409,1,0.0,34.0,1.1,28.0,-2.2,0.0,0.0,0.0
1,nwisia.01.02400517,2024-01-01,USGS-05474000,"Skunk River at Augusta, IA",Number of sampling points,1.0,count,Actual,Stream,7080107,...,14.179409,1,0.0,34.0,1.1,28.0,-2.2,0.0,0.0,0.0
2,nwisia.01.02400551,2024-02-26,USGS-05451210,"South Fork Iowa River NE of New Providence, IA",Stream width measure,30.0,ft,Actual,Stream,7080207,...,27.230696,57,11.0,62.0,16.7,25.0,-3.9,0.0,0.0,0.0


## Step 8: Summary and Save

In [10]:
print('=== Merged Dataset Summary ===')
print(f'Total rows:              {len(df_merged):,}')
print(f'Unique EPA locations:    {df_merged["MonitoringLocationIdentifier"].nunique():,}')
print(f'Unique climate stations: {df_merged["climate_station"].nunique():,}')
print(f'Date range:              {df_merged["ActivityStartDate"].min().date()} to {df_merged["ActivityStartDate"].max().date()}')
print(f'Climate match rate:      {climate_match_rate:.1f}%')
print(f'\nColumns: {df_merged.columns.tolist()}')
print(f'\nMissing values:')
print(df_merged.isnull().sum()[df_merged.isnull().sum() > 0])

=== Merged Dataset Summary ===
Total rows:              69,482
Unique EPA locations:    621
Unique climate stations: 110
Date range:              2024-01-01 to 2025-12-25
Climate match rate:      99.3%

Columns: ['ActivityIdentifier', 'ActivityStartDate', 'MonitoringLocationIdentifier', 'MonitoringLocationName', 'CharacteristicName', 'ResultMeasureValue', 'ResultMeasure/MeasureUnitCode', 'ResultValueTypeName', 'MonitoringLocationTypeName', 'HUCEightDigitCode', 'LatitudeMeasure', 'LongitudeMeasure', 'StateCode', 'CountyCode', 'climate_station', 'climate_station_name', 'distance_to_climate_station_km', 'doy', 'gdd_40_86', 'high', 'highc', 'low', 'lowc', 'precip', 'snow', 'snowd']

Missing values:
ResultMeasure/MeasureUnitCode    6067
high                              495
highc                             495
low                               495
lowc                              495
precip                            200
snow                             2567
snowd                         

In [11]:
out_path = '../../data/tabular/merged'
os.makedirs(out_path, exist_ok=True)
out_file = f'{out_path}/epa-climate-merged.csv'
df_merged.to_csv(out_file, index=False)
print(f'Saved {len(df_merged):,} rows → {out_file}')

Saved 69,482 rows → ../../data/tabular/merged/epa-climate-merged.csv


In [12]:
df_merged.head(10)

,ActivityIdentifier,ActivityStartDate,MonitoringLocationIdentifier,MonitoringLocationName,CharacteristicName,ResultMeasureValue,ResultMeasure/MeasureUnitCode,ResultValueTypeName,MonitoringLocationTypeName,HUCEightDigitCode,...,distance_to_climate_station_km,doy,gdd_40_86,high,highc,low,lowc,precip,snow,snowd
0,nwisia.01.02400517,2024-01-01,USGS-05474000,"Skunk River at Augusta, IA","Temperature, water",0.00,deg C,Actual,Stream,7080107,...,14.179409,1,0.0,34.0,1.1,28.0,-2.2,0.0,0.0,0.0
1,nwisia.01.02400517,2024-01-01,USGS-05474000,"Skunk River at Augusta, IA",Number of sampling points,1.00,count,Actual,Stream,7080107,...,14.179409,1,0.0,34.0,1.1,28.0,-2.2,0.0,0.0,0.0
2,nwisia.01.02400551,2024-02-26,USGS-05451210,"South Fork Iowa River NE of New Providence, IA",Stream width measure,30.00,ft,Actual,Stream,7080207,...,27.230696,57,11.0,62.0,16.7,25.0,-3.9,0.0,0.0,0.0
3,nwisia.01.02400551,2024-02-26,USGS-05451210,"South Fork Iowa River NE of New Providence, IA","Temperature, water",4.70,deg C,Actual,Stream,7080207,...,27.230696,57,11.0,62.0,16.7,25.0,-3.9,0.0,0.0,0.0
4,nwisia.01.02400551,2024-02-26,USGS-05451210,"South Fork Iowa River NE of New Providence, IA","Temperature, air",8.90,deg C,Actual,Stream,7080207,...,27.230696,57,11.0,62.0,16.7,25.0,-3.9,0.0,0.0,0.0
5,nwisia.01.02400551,2024-02-26,USGS-05451210,"South Fork Iowa River NE of New Providence, IA",Barometric pressure,726.00,mm/Hg,Actual,Stream,7080207,...,27.230696,57,11.0,62.0,16.7,25.0,-3.9,0.0,0.0,0.0
6,nwisia.01.02400551,2024-02-26,USGS-05451210,"South Fork Iowa River NE of New Providence, IA","Stream flow, instantaneous",25.00,ft3/s,Actual,Stream,7080207,...,27.230696,57,11.0,62.0,16.7,25.0,-3.9,0.0,0.0,0.0
7,nwisia.01.02400551,2024-02-26,USGS-05451210,"South Fork Iowa River NE of New Providence, IA",Number of sampling points,10.00,count,Actual,Stream,7080207,...,27.230696,57,11.0,62.0,16.7,25.0,-3.9,0.0,0.0,0.0
8,nwisia.01.02400551,2024-02-26,USGS-05451210,"South Fork Iowa River NE of New Providence, IA","Height, gage",2.12,ft,Actual,Stream,7080207,...,27.230696,57,11.0,62.0,16.7,25.0,-3.9,0.0,0.0,0.0
9,nwisia.01.02400551,2024-02-26,USGS-05451210,"South Fork Iowa River NE of New Providence, IA",Specific conductance,579.00,uS/cm @25C,Actual,Stream,7080207,...,27.230696,57,11.0,62.0,16.7,25.0,-3.9,0.0,0.0,0.0


## (Optional) Station-to-Station Mapping Reference

In [13]:
# Save the EPA → climate station mapping for reference
df_epa_locs[['MonitoringLocationIdentifier', 'LatitudeMeasure', 'LongitudeMeasure',
              'climate_station', 'climate_station_name', 'distance_to_climate_station_km']].to_csv(
    f'{out_path}/epa-to-climate-station-map.csv', index=False
)
print(f'Saved station mapping → {out_path}/epa-to-climate-station-map.csv')
df_epa_locs.sort_values('distance_to_climate_station_km').head(10)

Saved station mapping → ../../data/tabular/merged/epa-to-climate-station-map.csv


,MonitoringLocationIdentifier,LatitudeMeasure,LongitudeMeasure,climate_station,climate_station_name,distance_to_climate_station_km
335,21IOWA_WQX-22040001,40.824972,-92.894015,IA6910,RATHBUN DAM,0.168489
293,21IOWA_WQX-10490004,42.070737,-90.695404,IA5131,MAQUOKETA-2-W,0.517581
53069,EPA_R7_WQX-01-NMQ-90,42.299388,-91.012551,IA1257,CASCADE,0.831082
595,21IOWA_WQX-22890004,40.711232,-91.970015,IA4389,KEOSAUQUA,0.974959
540,21IOWA_WQX-22240002,42.022643,-95.324804,IA2171,DENISON,1.001519
407,21IOWA_WQX-21890001,40.709206,-91.971532,IA4389,KEOSAUQUA,1.212267
2526,21IOWA_WQX-22010001,41.298198,-94.481333,IA3438,GREENFIELD,1.276003
660,21IOWA_WQX-22790004,41.731706,-92.732963,IA3473,GRINNELL-3-SW,1.342698
458,21IOWA_WQX-10890001,40.728232,-91.960615,IA4389,KEOSAUQUA,1.388130
89,USGS-05490500,40.727806,-91.959614,IA4389,KEOSAUQUA,1.444669
